# Testing Deployment TensorFlow Serving
Notebook ini menguji deployment Railway yang memakai SavedModel hasil TFX Pusher. Pengujian mencakup status model, metadata signature `examples: DT_STRING`, dan prediction menggunakan serialized `tf.Example` Base64.

In [ ]:
import base64, json
import pandas as pd
import requests
import tensorflow as tf

BASE_URL = "https://mlops-dicoding-reza-production.up.railway.app"
MODEL_NAME = "breast_cancer_model"
STATUS_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}"
METADATA_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}/metadata"
PREDICT_URL = f"{BASE_URL}/v1/models/{MODEL_NAME}:predict"

status_response = requests.get(STATUS_URL, timeout=60)
print("GET", STATUS_URL)
print("HTTP status:", status_response.status_code)
print(json.dumps(status_response.json(), indent=2))
status_response.raise_for_status()

metadata_response = requests.get(METADATA_URL, timeout=60)
print("\nGET", METADATA_URL)
print("HTTP status:", metadata_response.status_code)
metadata_json = metadata_response.json()
print(json.dumps(metadata_json, indent=2)[:8000])
metadata_response.raise_for_status()

metadata_text = json.dumps(metadata_json)
assert "DT_STRING" in metadata_text, "Expected DT_STRING serving input"
assert "serving_default_examples:0" in metadata_text, "Expected examples input tensor"
print("\nMetadata check PASSED: examples / DT_STRING")

## Prediction request
Satu baris dataset diubah menjadi `tf.train.Example`, diserialisasi, lalu dikirim sebagai Base64 melalui `instances[0].examples.b64`.

In [ ]:
row = pd.read_csv("data/breast_cancer.csv").drop(columns=["label"]).iloc[0]
feature_map = {
    str(name): tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))
    for name, value in row.items()
}
example = tf.train.Example(features=tf.train.Features(feature=feature_map))
serialized_b64 = base64.b64encode(example.SerializeToString()).decode("ascii")
payload = {"instances": [{"examples": {"b64": serialized_b64}}]}

prediction_response = requests.post(PREDICT_URL, json=payload, timeout=60)
print("POST", PREDICT_URL)
print("Payload: serialized tf.Example -> Base64")
print("HTTP status:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=2))
prediction_response.raise_for_status()

prediction_json = prediction_response.json()
assert "predictions" in prediction_json, "Prediction response missing predictions"
print("\nRAILWAY PREDICTION TEST PASSED")